In [ ]:
import getpass, requests, pandas as pd
from urllib.parse import unquote

In [ ]:
KEY = unquote(getpass.getpass('공공데이터포털 인증키').strip())
TAGO = 'https://apis.data.go.kr/1613000/BusRouteInfoInqireService'

def rows(op, **p):
    out = []
    for pg in range(1, 20):
        r = requests.get(f'{TAGO}/{op}', params={'serviceKey': KEY, '_type': 'json', 'pageNo': pg,
                                                  'numOfRows': 100, **p}, timeout=60)
        r.raise_for_status()
        items = r.json().get('response', {}).get('body', {}).get('items', '')
        if not isinstance(items, dict):
            break
        it = items.get('item', [])
        it = [it] if isinstance(it, dict) else it
        if not it:
            break
        out += it
        if len(it) < 100:
            break
    return out

for sg, cc in [('김천시', '37030'), ('고창군', '35370')]:
    d = pd.DataFrame(rows('getRouteNoList', cityCode=cc))
    print(f'\n[{sg}] 노선 행 {len(d)} / 컬럼 {d.columns.tolist()}')
    for c in ['startvehicletime', 'endvehicletime']:
        if c in d.columns:
            print(f'  {c}: 값 있음 {d[c].notna().sum()}/{len(d)}, 예시 {d[c].dropna().astype(str).head(5).tolist()}')
    print(d.head(3).to_string())

In [ ]:
import re, numpy as np

In [ ]:
def hm(v):
    s = re.sub(r'\D', '', str(v))
    if len(s) < 3: return np.nan
    s = s.zfill(4)[:4]; h, m = int(s[:2]), int(s[2:])
    return h * 60 + m if h < 30 and m < 60 else np.nan

tt = []
for sg, cc in [('김천시', '37030'), ('고창군', '35370')]:
    d = pd.DataFrame(rows('getRouteNoList', cityCode=cc)); d['sigun'] = sg
    tt.append(d[['sigun', 'routeid', 'routeno', 'startvehicletime', 'endvehicletime']])
tt = pd.concat(tt)
tt['f'], tt['l'] = tt['startvehicletime'].map(hm), tt['endvehicletime'].map(hm)
tt['routeno'] = tt['routeno'].astype(str)

gj = df[df['sigun'] == '공주시'].drop_duplicates('route_id')[['sigun', 'route_id', 'route_no', 'first_time', 'last_time']].copy()
gj['f'], gj['l'] = gj['first_time'].map(hm), gj['last_time'].map(hm)
gj = gj.rename(columns={'route_id': 'routeid', 'route_no': 'routeno'})

times = pd.concat([tt[['sigun', 'routeid', 'routeno', 'f', 'l']], gj[['sigun', 'routeid', 'routeno', 'f', 'l']]])
times['single'] = times['f'] == times['l']
print('=== 시군별 시간표 요약 (route_id 단위) ===')
print(times.groupby('sigun').agg(건수=('routeid', 'size'), 막차결측=('l', lambda s: int(s.isna().sum())),
                                  첫차막차같음=('single', 'sum')).to_string())

lt = times.groupby(['sigun', 'routeno']).agg(first=('f', 'min'), last=('l', 'max'), n_ids=('routeid', 'nunique')).reset_index()
lt['rkey'] = lt['sigun'] + '|' + lt['routeno']
lt = lt.set_index('rkey')
print('\n핵심 노선: ')
for k in ['김천시|85', '공주시|321']:
    if k in lt.index:
        r = lt.loc[k]; print(' ', k, '첫차', round(r['first'] / 60, 1), '시 / 막차', round(r['last'] / 60, 1), '시 / route_id', int(r['n_ids']), '개')

name = sub.set_index('adm_cd8')['읍면동명칭'].to_dict(); pp = sub.set_index('adm_cd8')['pop65'].to_dict()
out = []
for sg in T:
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: region_nodes.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    no = {k: v for k, v in nodes_all.items() if k.startswith(sg + '|')}
    b = level_for_regions(rn, hosp_nodes[sg], no)
    for a, (lv, dr) in b.items():
        ls = [lt.loc[k, 'last'] for k in dr if k in lt.index and pd.notna(lt.loc[k, 'last'])]
        fs = [lt.loc[k, 'first'] for k in dr if k in lt.index and pd.notna(lt.loc[k, 'first'])]
        out.append({'sigun': sg, '읍면동': name[a], 'pop65': int(pp[a]), 'label': LABEL[lv], 'n_direct': len(dr),
                    '직통 막차(시)': round(max(ls) / 60, 1) if ls else np.nan,
                    '직통 첫차(시)': round(min(fs) / 60, 1) if fs else np.nan})

In [ ]:
tm = pd.DataFrame(out)
print('\n=== 읍면동의 병원 직통 노선 막차(가장 늦은 것) 분포 ===')
print(tm.groupby('sigun')['직통 막차(시)'].describe()[['count', 'min', '25%', '50%', 'max']].round(1).to_string())
print('\n=== 직통 막차가 이른 읍면동 12곳 ===')
print(tm.sort_values('직통 막차(시)').head(12).to_string(index=False))
print('\n=== 노선 기반 후보 7곳의 시간 ===')
print(tm[tm['읍면동'].isin(['탄천면', '반포면', '구성면', '지례면', '부항면', '대덕면', '증산면'])].to_string(index=False))

In [ ]:
for sg, ln in [('김천시', '85'), ('공주시', '321')]:
    x = df[(df['sigun'] == sg) & (df['line'] == ln)].sort_values(['route_id', 'stop_seq'])
    print(f'\n===== {sg} {ln}번 / route_id {x["route_id"].nunique()}개 =====')
    for rid, g in x.groupby('route_id'):
        tm_ = times[times['routeid'] == rid]
        f_ = tm_['f'].iloc[0] if len(tm_) else np.nan; l_ = tm_['l'].iloc[0] if len(tm_) else np.nan
        print(f'  {rid}: {g["stop_name"].iloc[0]} → {g["stop_name"].iloc[-1]} | 정류장 {len(g)}개 | '
              f'첫차 {f_ / 60:.1f}시 막차 {l_ / 60:.1f}시' if pd.notna(f_) else
              f'  {rid}: {g["stop_name"].iloc[0]} → {g["stop_name"].iloc[-1]} | 정류장 {len(g)}개 | 시간 결측')
    inter = nodes_all[f'{sg}|{ln}'] & hosp_nodes[sg]
    print('  병원 300m 안 지나는 지점:', node_xy.set_index('node').loc[list(inter), 'name'].tolist())

In [ ]:
from collections import Counter
df2 = df.copy(); df2['line'] = df2['route_id'].astype(str)
nxy2, nall2 = build_network(df2)
rn2_all, hn2_all = assign_nodes(nxy2, sub, hg)
rid_no = df.drop_duplicates('route_id').set_index('route_id')['route_no'].astype(str).to_dict()
tl = times.drop_duplicates('routeid').set_index('routeid')['l'].to_dict()

In [ ]:
res, loss = [], []

for sg in T:
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: rn2_all.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    no = {k: v for k, v in nall2.items() if k.startswith(sg + '|')}
    p65 = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
    base, imp = simulate_line_removal(rn, hn2_all[sg], no, p65)
    for a, (lv, dr) in base.items():
        ls = [tl.get(k.split('|', 1)[1]) for k in dr]
        ls = [x for x in ls if x is not None and pd.notna(x)]
        res.append({'sigun': sg, '읍면동': name[a], 'pop65': int(pp[a]), 'label': LABEL[lv], 'n_direct': len(dr),
                    '직통 노선번호': ','.join(sorted({rid_no[k.split('|', 1)[1]] for k in dr}))[:30],
                    '직통 막차(시)': round(max(ls) / 60, 1) if ls else np.nan})
    for _, r in imp[(imp['n_lost_direct'] > 0) | (imp['n_lost_any'] > 0)].iterrows():
        rid = r['rkey'].split('|', 1)[1]
        loss.append({'sigun': sg, '노선번호': rid_no[rid], 'route_id': rid, '직통잃는읍면수': r['n_lost_direct'],
                     '65세이상': int(r['pop65_lost_direct']), '연결상실읍면수': r['n_lost_any']})

In [ ]:
R = pd.DataFrame(res)
print('=== route_id 단위(엄격) 병원 연결 수준(읍면동 수) ===')
print(R.groupby(['sigun', 'label']).size().unstack(fill_value=0).to_string())
print('\n=== 직통이 아니거나 직통 노선 1개뿐인 읍면동 ===')
print(R[(R['label'] != '직통') | (R['n_direct'] == 1)].to_string(index=False))
print('\n=== 노선(route_id) 하나를 빼면 직통/연결을 잃는 경우 ===')
print(pd.DataFrame(loss).sort_values('65세이상', ascending=False).to_string(index=False) if loss else '없음')

In [ ]:
import json

TAGO_ = 'https://apis.data.go.kr/1613000/BusRouteInfoInqireService'
for rid in ['GCB47201709', 'GCB47201465', 'GCB47201567']:
    r = requests.get(f'{TAGO_}/getRouteInfoIem', params={'serviceKey': KEY, '_type': 'json',
                     'cityCode': '37030', 'routeId': rid}, timeout=60)
    body = r.json().get('response', {}).get('body', {}).get('items', {})
    print(rid, r.status_code); print(json.dumps(body, ensure_ascii=False, indent=1)[:900], '\n')

In [ ]:
#상하행구분

ts = pd.read_json('/content/tago_stops.jsonl', lines=True, dtype=False)
for rid in ['GCB47201709', 'GCB47201465']:
    x = ts[ts['_rid'] == rid]
    print(rid, '정류장', len(x), '/ updowncd 분포', x['updowncd'].value_counts(dropna=False).to_dict() if 'updowncd' in x else '컬럼 없음',
          '/ nodeord 최대', x['nodeord'].max())

# 지표 강화

In [ ]:
from shapely.ops import unary_union
L = pd.DataFrame(loss)
R2 = R.merge(conn[['sigun', '읍면동', 'adm_cd8', 'd_near_m', 'hosp_km', 'ratio65', 'n_nodes_in']], on=['sigun', '읍면동'])
def cat2(r):
    if r['n_nodes_in'] == 0: return '노선 데이터 미확인'
    if r['label'] == '환승1회': return '환승 필요'
    if r['label'] == '직통' and r['n_direct'] == 1: return '단일 노선 의존'
    return '그 외'
R2['cat'] = R2.apply(cat2, axis=1)
print('=== 엄격 기준 유형별 읍면동 수 ===')
print(R2.groupby(['sigun', 'cat']).size().unstack(fill_value=0).to_string())

In [ ]:
if 'cover500' not in sub.columns:
    sub['cover500'] = sub.geometry.intersection(unary_union(stg.geometry.buffer(500))).area / sub.geometry.area
c = R2.merge(sub[['adm_cd8', 'cover500']], on='adm_cd8')
c['r_near'] = c['d_near_m'].rank(ascending=False, method='min'); c['r_cov'] = c['cover500'].rank(method='min')
c['r_hosp'] = c['hosp_km'].rank(ascending=False, method='min'); c['r_old'] = c['ratio65'].rank(ascending=False, method='min')
V = c[c['cat'].isin(['단일 노선 의존', '환승 필요'])]
print('\n=== 후보와 기존 지표 순위 (52개 중) ===')
print(V[['sigun', '읍면동', 'pop65', 'cat', 'r_near', 'r_cov', 'r_hosp', 'r_old']].round(0).to_string(index=False))
for k in (5, 10, 15, 20):
    o = {nm: f"{int((V[col] <= k).sum())} ({len(V) * k / len(c):.1f})" for col, nm in
         [('r_near', '정류장거리'), ('r_cov', '커버율'), ('r_hosp', '병원거리'), ('r_old', '고령비율')]}
    a3 = int(((V['r_near'] <= k) | (V['r_cov'] <= k) | (V['r_hosp'] <= k)).sum())
    print(f'k={k}:', o, f'| 거리계 3개 중 하나라도 상위k: {a3}/{len(V)}')

In [ ]:
S = sub.merge(R2[['adm_cd8', 'cat']], on='adm_cd8')
fig, axs = plt.subplots(1, 3, figsize=(13, 5.6))
for ax, sg in zip(axs, T):
    d = S[S['시군구명칭'] == sg]
    d.plot(ax=ax, color=d['cat'].map(COL).tolist(), edgecolor='white', linewidth=0.8)
    na = d[d['cat'] == '노선 데이터 미확인']
    if len(na): na.plot(ax=ax, color=C_NA, edgecolor=INK2, linewidth=0.8, hatch='///')
    hg[hg['시군구'] == sg].plot(ax=ax, color=INK, marker='P', markersize=50, zorder=5)
    for _, r in d[d['cat'] != '그 외'].iterrows():
        p = r.geometry.representative_point()
        ax.annotate(r['읍면동명칭'], (p.x, p.y), ha='center', va='center', fontsize=8.5, color=INK, zorder=6,
                    bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.85))
    if (d['cat'] != '그 외').sum() == 0:
        ax.text(0.5, -0.01, '해당 읍면 없음', transform=ax.transAxes, ha='center', va='top', fontsize=9.5, color=INK2)
    ax.set_title(sg, fontsize=12, color=INK, y=1.0); ax.set_axis_off()
handles = [Patch(fc=C_SINGLE, label='병원 직통이 운행(노선) 1개뿐'), Patch(fc=C_TRANS, label='병원까지 환승 필요'),
           Patch(fc=C_NA, ec=INK2, hatch='///', label='노선 데이터 미확인(아포읍)'),
           Patch(fc=C_OTHER, label='그 외 (병원 직통 2개 이상)'),
           Line2D([], [], marker='P', color='w', markerfacecolor=INK, markersize=9, label='병원(병원·종합병원)')]
fig.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 0.06), ncol=3, frameon=False, fontsize=9)
fig.text(0.01, 0.01, '개별 운행(route_id) 단위, 병원 반경 300m. 경계 2025-06-30, 정류장 2025-10-31, 병원 2026-10-12, 인구 2026-09-30, 노선 API 수집본.', fontsize=7.5, color=INK2)
fig.tight_layout(rect=(0, 0.14, 1, 1))
fig.savefig('/content/outputs/fig1_map.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
items = []
for _, r in L[L['직통잃는읍면수'] > 0].sort_values('65세이상').iterrows():
    sg, rid = r['sigun'], r['route_id']
    e = sub[sub['시군구명칭'] == sg]
    rn = {a: rn2_all.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
    no = {k: v for k, v in nall2.items() if k.startswith(sg + '|')}
    b = level_for_regions(rn, hn2_all[sg], no)
    a2 = level_for_regions(rn, hn2_all[sg], {k: v for k, v in no.items() if k != f'{sg}|{rid}'})
    segs = [(name[x], int(pp[x])) for x in b if b[x][0] == 0 and a2[x][0] > 0]
    g = df[df['route_id'] == rid].sort_values('stop_seq')
    t_ = times[times['routeid'] == rid]
    hh = f"{int(t_['f'].iloc[0] // 60):02d}:{int(t_['f'].iloc[0] % 60):02d}" if len(t_) and pd.notna(t_['f'].iloc[0]) else '시간 결측'
    items.append((f"{sg[:2]} {r['노선번호']}번\n{g['stop_name'].iloc[0]}→{g['stop_name'].iloc[-1]}\n{hh} 출발", segs))
fig, ax = plt.subplots(figsize=(10, 2.4 + 1.2 * len(items)))
for i, (lab, segs) in enumerate(items):
    left = 0
    for nm, v in sorted(segs, key=lambda t: -t[1]):
        ax.barh(i, v, left=left, height=0.5, color=C_SINGLE, edgecolor='white', linewidth=2)
        ax.text(left + v / 2, i, f"{nm}\n{v:,}명", ha='center', va='center', fontsize=9, color='white', fontweight='bold')
        left += v
    ax.text(left + 40, i, f"합계 {left:,}명", va='center', fontsize=10, color=INK)
ax.set_yticks(range(len(items))); ax.set_yticklabels([t[0] for t in items], fontsize=10)
ax.set_xlabel('이 운행이 사라지면 병원 직통을 잃는 읍면의 65세 이상 인구(명)', color=INK2)
ax.set_xlim(0, max(sum(v for _, v in s) for _, s in items) * 1.25)
fig.text(0.02, -0.02, f"나머지 {len(imp_all) - len(items)}개 노선·운행은 하나를 빼도 직통을 잃는 읍면이 없음(0명). 개별 운행(route_id) 단위, 병원 반경 300m. 돌아오는 방향 운행은 데이터에 없음.", fontsize=8.5, color=INK2)
fig.savefig('/content/outputs/fig2_route_removal.png', bbox_inches='tight', facecolor='white'); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 6))
g0 = R2[R2['cat'] == '그 외']
ax.scatter(g0['d_near_m'], g0['hosp_km'], s=28, color='#b9b8b2', edgecolor='white', linewidth=0.8, zorder=2)
for k in ('단일 노선 의존', '환승 필요'):
    g1 = R2[R2['cat'] == k]
    ax.scatter(g1['d_near_m'], g1['hosp_km'], s=70, color=COL[k], edgecolor='white', linewidth=1.2, zorder=3, label=k)
na = R2[R2['cat'] == '노선 데이터 미확인']
ax.scatter(na['d_near_m'], na['hosp_km'], s=70, facecolor='white', edgecolor=INK2, linewidth=1.5, zorder=3, label='노선 데이터 미확인')
OFF = {'탄천면': (10, 4, 'left'), '구성면': (9, -13, 'left'), '지례면': (9, 6, 'left'), '부항면': (-9, 7, 'right'),
       '대덕면': (-9, 8, 'right'), '증산면': (-9, -15, 'right'), '아포읍': (-9, 9, 'right'),
       '감천면': (9, 6, 'left'), '조마면': (9, -13, 'left')}
for _, r in R2[R2['cat'] != '그 외'].iterrows():
    dx, dy, ha = OFF.get(r['읍면동'], (8, 6, 'left'))
    ax.annotate(f"{r['sigun'][:2]} {r['읍면동']}", (r['d_near_m'], r['hosp_km']), xytext=(dx, dy),
                textcoords='offset points', fontsize=9, color=INK, ha=ha)
ax.axvline(R2['d_near_m'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.axhline(R2['hosp_km'].median(), color='#d4d3ce', lw=1, ls='--', zorder=1)
ax.set_ylim(-1, 31)
ax.set_xlabel('가장 가까운 정류장까지 거리(m, 읍면동 대표점)', color=INK2); ax.set_ylabel('같은 시군 병원까지 직선거리(km)', color=INK2)
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.text(0.01, -0.12, '점선은 52개 읍면동 중앙값. 회색 점은 나머지 읍면동.', transform=ax.transAxes, fontsize=8, color=INK2)
fig.savefig('/content/outputs/fig3_stop_vs_hospital.png', bbox_inches='tight', facecolor='white'); plt.show()

- 민감도 강화

In [ ]:
import itertools
from collections import Counter

nm = sub.set_index('adm_cd8')['읍면동명칭'].to_dict()
df2 = df.copy()
df2['line'] = df2['route_id'].astype(str)
## 엄격 기준: 개별 운행 단위

KEY = [('공주시','탄천면'), ('김천시','구성면'), ('김천시','지례면'), ('김천시','부항면'),
       ('김천시','조마면'), ('김천시','감천면'), ('김천시','대덕면'), ('김천시','증산면')]

In [ ]:
summ, status = [], {}

In [ ]:
for mr in [30, 60, 100]:
    node_xy, nodes_all = build_network(df2, mr)
    for hr in [300, 500]:
        region_nodes, hosp_nodes = assign_nodes(node_xy, sub, hg, hr)
        tag = f'병합{mr}·병원{hr}'
        for sg in T:
            e = sub[sub['시군구명칭'] == sg]
            rn = {a: region_nodes.get(sg, {}).get(a, set()) for a in e['adm_cd8']}
            hn = hosp_nodes.get(sg, set())
            nodes_of = {k: v for k, v in nodes_all.items() if k.startswith(sg + '|')}
            pop65 = e.set_index('adm_cd8')['pop65'].fillna(0).to_dict()
            base, imp = simulate_line_removal(rn, hn, nodes_of, pop65)
            cnt = Counter(v[0] for v in base.values())
            single = [a for a, (lv, d) in base.items() if lv == 0 and len(d) == 1]
            summ.append({'설정': tag, '시군': sg, '직통': cnt[0], '환승1': cnt[1], '불가': cnt[9],
                         '단일운행의존': len(single),
                         '단일의존_65+': int(sum(pop65.get(a, 0) for a in single)),
                         '직통잃는운행수': int((imp['n_lost_direct'] > 0).sum()),
                         '연결잃는운행수': int((imp['n_lost_any'] > 0).sum())})
            for a, (lv, d) in base.items():
                s = '단일' if (lv == 0 and len(d) == 1) else '직통(복수)' if lv == 0 else '환승' if lv == 1 else '불가/미확인'
                status[(sg, nm[a], tag)] = s

In [ ]:
S = pd.DataFrame(summ)
print(S.to_string(index=False))
print('\n=== 핵심 8개 읍면 상태 ===')
P = pd.DataFrame({tag: {f'{sg} {n}': status.get((sg, n, tag), '-') for sg, n in KEY}
                  for tag in S['설정'].unique()})
print(P.to_string())
S.to_csv('/content/outputs/sensitivity_strict.csv', index=False, encoding='utf-8-sig')